# Plately — training on Colab

Trains the twelve-class food classifier on a free T4 GPU. A full run takes
a few minutes here versus an hour or more on a laptop CPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Your own photographs are the one input Colab cannot fetch. Upload the
`raw_nigerian/` folder to your Drive first — it is expected at
`MyDrive/plately/raw_nigerian/`, with one subfolder per dish.


## 1. Confirm the GPU


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

# If this says 'command not found', the runtime is still on CPU:
# Runtime -> Change runtime type -> T4 GPU, then run this cell again.


## 2. Mount Drive

Holds your photographs on the way in, and the trained model on the way out.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PHOTOS = '/content/drive/MyDrive/plately/raw_nigerian'

import os
assert os.path.isdir(PHOTOS), f'No photos at {PHOTOS} — upload raw_nigerian/ to Drive first.'
for dish in sorted(os.listdir(PHOTOS)):
    count = len(os.listdir(os.path.join(PHOTOS, dish)))
    print(f'  {dish:<16} {count:>4} images')


## 3. Get the project

Clones the code. Your photographs are not in the repository — they come from Drive, mounted above.


### Clone from GitHub


In [ ]:
REPO = 'https://github.com/Olamidehash1234/Plately.git'

!rm -rf /content/project
!git clone --depth 1 $REPO /content/project
%cd /content/project

import os
assert os.path.isfile('backend/requirements-ml.txt'), 'Clone failed — check the repo URL.'
print('Project ready:', os.getcwd())


## 4. Install TensorFlow and friends

Colab ships TensorFlow already; this pins the versions the project expects
and adds scikit-learn and matplotlib for the evaluation reports.


In [ ]:
!pip install -q -r backend/requirements-ml.txt
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE — check the runtime type')


## 5. Fetch the six Food-101 classes

Streams the 5GB archive on Google's network and keeps only our six
categories, about 300MB. Two minutes or so.


In [ ]:
!python ml/fetch_food101.py


## 6. Build the dataset

Shuffles with a fixed seed and splits 75/25. `--limit-per-class` caps the
Food-101 classes so they cannot swamp the Nigerian ones — keep it near the
number of photographs you have per dish.


In [ ]:
!python ml/prepare_data.py \
    --food101 ml/raw_food101 \
    --nigerian '{PHOTOS}' \
    --limit-per-class 250 \
    --clean


## 7. Train

Phase 1 trains a new classifier head on frozen MobileNetV2 features.
Phase 2 unfreezes the top 30 layers at a much lower learning rate.

Add `--no-fine-tune` for a quick first pass, or `--epochs N` to lengthen
phase 1.


In [ ]:
!python ml/train.py


## 8. Evaluate

Runs the held-out test split and writes the numbers Chapter 4 and Chapter 5
need. Read the confusion matrix, not just the accuracy: it tells you *how*
the model fails, and confusions between amala, eba and pounded yam are a
data problem, not a training-length one.


In [ ]:
!python ml/evaluate.py


In [ ]:
import json
from IPython.display import Image, display

metrics = json.load(open('ml/artifacts/reports/metrics.json'))
print(json.dumps(metrics, indent=2)[:2000])
print(open('ml/artifacts/reports/classification_report.txt').read())
display(Image('ml/artifacts/reports/confusion_matrix.png'))
display(Image('ml/artifacts/reports/training_curves.png'))


## 9. Export the deployable model

Converts to TFLite and checks the converted model still agrees with the
original on the test set. This is what the 349MB production image serves,
instead of a 1.5GB TensorFlow one.


In [ ]:
!python ml/export_tflite.py


## 10. Take the results home

Copies everything back to Drive. Do this as soon as training finishes —
Colab disconnects idle sessions and the filesystem goes with it.


In [ ]:
!mkdir -p /content/drive/MyDrive/plately/artifacts
!cp -r ml/artifacts/* /content/drive/MyDrive/plately/artifacts/
!ls -lh /content/drive/MyDrive/plately/artifacts

# Then, on your own machine:
#   cp -r ~/Drive/plately/artifacts/* 'Nutrition Project/ml/artifacts/'
#   curl http://localhost:8000/classify/status
